In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Environment Setup & Initialization


In [3]:
!pip install pandas numpy torchvision transformers gdown fasttext langid scikit-learn tqdm -q
import os
%cd /content
if not os.path.exists('SMTPD'):
    !git clone https://github.com/zhuwei321/SMTPD.git

%cd /content/SMTPD
print("Environment setup successful!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 4.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 49.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
/content
Cloning into 'SMTPD'...
remote: Enumerating objects: 51, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 51 (delta 23), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (51/51), 349.57 KiB | 1.90 MiB/s, done.
Resolving deltas: 100% (23/23), done.
/content/SMTPD
Environment setup successful!


Data Acquisition & Extraction

In [4]:
import gdown
import os

# Create directory for data
data_dir = '/content/SMTPD/data_source'
os.makedirs(data_dir, exist_ok=True)

# File IDs from the SMTPD dataset
csv_id = '12lVJPuQDWH1tMTZegYVu6XmG65JZW3Nu'
zip_id = '1ZbGIlQlAJ5C-6OOzbOGGtULf7SfdIAuV'

print("--- Downloading data (CSV & 9.25GB Images) ---")
# Download CSV file
gdown.download(f'https://drive.google.com/uc?id={csv_id}', os.path.join(data_dir, 'basic_view_pn.csv'), quiet=False)

# Download large image archive
print("\nDownloading large image file (9.25GB)...")
gdown.download(f'https://drive.google.com/uc?id={zip_id}', os.path.join(data_dir, 'img_yt.zip'), quiet=False)

# Extract and clean up storage immediately
print("\n--- Extracting images ---")
%cd {data_dir}
!unzip -q img_yt.zip && rm img_yt.zip
print("Data preparation complete!")

--- Downloading data (CSV & 9.25GB Images) ---


Downloading...
From (original): https://drive.google.com/uc?id=12lVJPuQDWH1tMTZegYVu6XmG65JZW3Nu
From (redirected): https://drive.google.com/uc?id=12lVJPuQDWH1tMTZegYVu6XmG65JZW3Nu&confirm=t&uuid=f1285b66-2438-4355-b08b-4312f41a6fdc
To: /content/SMTPD/data_source/basic_view_pn.csv
100%|██████████| 609M/609M [00:08<00:00, 68.7MB/s]


Downloading...
From (original): https://drive.google.com/uc?id=1ZbGIlQlAJ5C-6OOzbOGGtULf7SfdIAuV
From (redirected): https://drive.google.com/uc?id=1ZbGIlQlAJ5C-6OOzbOGGtULf7SfdIAuV&confirm=t&uuid=d31805dd-c665-4b37-8c2a-ee2c3a31a801
To: /content/SMTPD/data_source/img_yt.zip
100%|██████████| 9.93G/9.93G [02:14<00:00, 73.7MB/s]



--- Extracting images ---
/content/SMTPD/data_source
Data preparation complete!


Dataset Subsampling (30% Dataset)

In [5]:
import pandas as pd
import os

original_csv = '/content/SMTPD/data_source/basic_view_pn.csv'
sampled_csv = '/content/SMTPD/data_source/basic_view_pn_30.csv'

print("Reading the original dataset...")
df = pd.read_csv(original_csv)

# Randomly sample 30% of the data (frac=0.3)
df_30 = df.sample(frac=0.3, random_state=42)

# Save to a new CSV file
print("Saving the 30% sampled dataset ...")
df_30.to_csv(sampled_csv, index=False)

print(f"Complete! Total original samples: {len(df)}")
print(f"Total samples to be used for training (30%): {len(df_30)}")

Reading the original dataset...
Saving the 30% sampled dataset ...
Complete! Total original samples: 282481
Total samples to be used for training (30%): 84744


Codebase Patching & Model Configuration

In [6]:
%cd /content/SMTPD

# --- Patch 1: Update BERT model path to HuggingFace online version ---
with open('smp_model.py', 'r') as f:
    content = f.read()
content = content.replace("'../bert_multilingual'", "'bert-base-multilingual-cased'")
with open('smp_model.py', 'w') as f:
    f.write(content)

# --- Patch 2: ENABLE EARLY POPULARITY FEATURE (Crucial) ---
with open('smp_data.py', 'r') as f:
    data_content = f.read()
# Toggle flag from self.p_i = 0 (disabled) to self.p_i = 1 (enabled)
data_content = data_content.replace("self.p_i = 0", "self.p_i = 1")
with open('smp_data.py', 'w') as f:
    f.write(data_content)

print("Source code successfully patched and Early Popularity ENABLED!")

/content/SMTPD
Source code successfully patched and Early Popularity ENABLED!


In [7]:
# Change working directory to the project root
%cd /content/SMTPD

# Open main.py and read its contents
with open('main.py', 'r') as f:
    content = f.read()

# Remove the verbose=True parameter from the ReduceLROnPlateau scheduler
content = content.replace("verbose=True,", "")
content = content.replace("verbose=True", "")

# Write the modified content back
with open('main.py', 'w') as f:
    f.write(content)

print("PyTorch Scheduler successfully patched!")

/content/SMTPD
PyTorch Scheduler successfully patched!


In [8]:
import os

file_path = '/content/SMTPD/smp_model.py'

# Read the contents of smp_model.py
with open(file_path, 'r') as f:
    content = f.read()

# Replace local path with online path (handles both single and double quotes)
content = content.replace("'../bert_multilingual'", "'bert-base-multilingual-cased'")
content = content.replace('"../bert_multilingual"', '"bert-base-multilingual-cased"')

# Overwrite the file
with open(file_path, 'w') as f:
    f.write(content)

print("smp_model.py successfully patched!")

smp_model.py successfully patched!


Model Training Execution

In [ ]:
%cd /content/SMTPD
!mkdir -p ./checkpoint

!python main.py \
    --train True \
    --K_fold 0 \
    --epochs 15 \
    --batch_size 64 \
    --lr 0.001 \
    --images_dir "./data_source/img_yt" \
    --data_files "./data_source/basic_view_pn_30.csv" \
    --ckpt_path "./checkpoint"

/content/SMTPD
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/loss.py:44: UserWarning: size_average and reduce args will be deprecated, please use reduction='mean' instead.
  self.reduction: str = _Reduction.legacy_get_string(size_average, reduce)
./data_source/basic_view_pn_30.csv
Downloading: "https://download.pytorch.org/models/resnet101-63fe2227.pth" to /root/.cache/torch/hub/checkpoints/resnet101-63fe2227.pth
100% 171M/171M [00:01<00:00, 174MB/s]
tokenizer_config.json: 100% 49.0/49.0 [00:00<00:00, 199kB/s]
vocab.txt: 996kB [00:00, 5.11MB/s]
tokenizer.json: 1.96MB [00:00, 7.86MB/s]
config.json: 100% 625/625 [00:00<00:00, 2.72MB/s]
model.safetensors: 100% 714M/714M [00:07<00:00, 101MB/s]
Loading weights: 100% 199/199 [00:00<00:00, 1101.87it/s, Materializing param=pooler.dense.weight]
100% 1059/1059 [28:33<00:00,  1.62s/it]
=====Epoch 1 averaged training loss: 1.140810=====
=====Epoch 1 train result=====
[1.5981512368924429, 1.6871479421099242, 1.7149769748469623, 1.6755259